In [2]:
import os
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load metadata
metadata_path = "./archive/UrbanSound8K.csv"
metadata = pd.read_csv(metadata_path)

# Create file paths and labels
file_paths = ["./archive/fold{}/{}".format(row['fold'], row["slice_file_name"]) for _, row in metadata.iterrows()]
labels = metadata["classID"].values

# Split into train and test sets
train_file_paths, test_file_paths, y_train, y_test = train_test_split(file_paths, labels, test_size=0.2, random_state=42)

# Feature extraction function
def extract_features(file_path, n_mfcc=40, max_pad_len=174):
    try:
        audio, sample_rate = librosa.load(file_path, sr=22050)
        
        # Handle n_fft warning by ensuring n_fft <= len(audio)
        n_fft = min(2048, len(audio))
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=n_mfcc, n_fft=n_fft)

        # Pad or truncate to fixed shape
        if mfccs.shape[1] < max_pad_len:
            pad_width = max_pad_len - mfccs.shape[1]
            mfccs = np.pad(mfccs, ((0, 0), (0, pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :max_pad_len]

        return mfccs.T  # Shape: (174, 40)
    
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Apply feature extraction
X_train_features = np.array([f for f in (extract_features(fp) for fp in train_file_paths) if f is not None])
X_test_features = np.array([f for f in (extract_features(fp) for fp in test_file_paths) if f is not None])

# Ensure no empty feature sets
if X_train_features.size == 0 or X_test_features.size == 0:
    raise ValueError("Feature extraction failed for all files. Check file paths and processing logic.")

# Normalize features
scaler = StandardScaler()
X_train_features = scaler.fit_transform(X_train_features.reshape(-1, X_train_features.shape[-1])).reshape(X_train_features.shape)
X_test_features = scaler.transform(X_test_features.reshape(-1, X_test_features.shape[-1])).reshape(X_test_features.shape)

# Ensure labels are one-hot encoded
y_train = to_categorical(y_train[:len(X_train_features)], num_classes=10)
y_test = to_categorical(y_test[:len(X_test_features)], num_classes=10)

# Transformer-based model
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation="relu"),
            tf.keras.layers.Dense(embed_dim)
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, inputs, training=False):  # Fixed: Ensure training argument is included
        attn_output = self.att(inputs, inputs, training=training)  # Pass training
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Model function
def build_transformer_model(input_shape, embed_dim=64, num_heads=4, ff_dim=128, num_classes=10):
    inputs = tf.keras.Input(shape=input_shape)
    
    # Linear embedding layer
    x = tf.keras.layers.Dense(embed_dim)(inputs)
    
    # Transformer blocks
    x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)
    x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)

    # Global average pooling
    x = tf.keras.layers.GlobalAveragePooling1D()(x)

    # Dense layers
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="categorical_crossentropy", metrics=["accuracy"])
    
    return model

# Build the transformer model
input_shape = (X_train_features.shape[1], X_train_features.shape[2])
transformer_model = build_transformer_model(input_shape)

# Train the model
history = transformer_model.fit(X_train_features, y_train, validation_data=(X_test_features, y_test),
                                epochs=30, batch_size=32, verbose=1)

# Evaluate the model
score = transformer_model.evaluate(X_test_features, y_test)
print(f"Transformer Model Accuracy: {score[1] * 100:.2f}%")



Epoch 1/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 156s 539ms/step - accuracy: 0.4473 - loss: 1.5877 - val_accuracy: 0.7201 - val_loss: 0.8542
Epoch 2/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 92s 420ms/step - accuracy: 0.7284 - loss: 0.8169 - val_accuracy: 0.7894 - val_loss: 0.6121
Epoch 3/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 117s 534ms/step - accuracy: 0.8158 - loss: 0.5490 - val_accuracy: 0.8414 - val_loss: 0.4949
Epoch 4/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 132s 604ms/step - accuracy: 0.8703 - loss: 0.3989 - val_accuracy: 0.8569 - val_loss: 0.4590
Epoch 5/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 97s 439ms/step - accuracy: 0.8958 - loss: 0.3057 - val_accuracy: 0.8558 - val_loss: 0.4278
Epoch 6/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 86s 394ms/step - accuracy: 0.9133 - loss: 0.2733 - val_accuracy: 0.8947 - val_loss: 0.3291
Epoch 7/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 87s 397ms/step - accuracy: 0.9253 - loss: 0.2187 - val_accuracy: 0.8643 - val_loss: 0.4522
Epoch 8/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 133s 607ms/step - accuracy: 0.9246 - lo

In [ ]:
# transformer model with cnn and using mel-log frequency
import os
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import random
# Load metadata
metadata_path = "./archive/UrbanSound8K.csv"
metadata = pd.read_csv(metadata_path)

# Create file paths and labels
file_paths = ["./archive/fold{}/{}".format(row['fold'], row["slice_file_name"]) for _, row in metadata.iterrows()]
labels = metadata["classID"].values

# Train-test split
train_file_paths, test_file_paths, y_train, y_test = train_test_split(file_paths, labels, test_size=0.2, random_state=42)
def augment_audio(audio, sample_rate):
    """Apply random noise, pitch shift, and time stretch."""
    if random.random() < 0.5:
        audio = audio + 0.005 * np.random.randn(len(audio))  # Add noise
    if random.random() < 0.5:
        audio = librosa.effects.pitch_shift(y=audio, sr=sample_rate, n_steps=random.choice([-2, -1, 1, 2]))  # Pitch shift
    if random.random() < 0.5:
        audio = librosa.effects.time_stretch(y=audio, rate=random.uniform(0.8, 1.2))  # Time stretch
    return audio


def extract_features(file_path, n_mels=64, max_pad_len=174):
    """Extract log-mel spectrogram features."""
    try:
        audio, sample_rate = librosa.load(file_path, sr=22050)
        audio = augment_audio(audio, sample_rate)  # Apply augmentation
        
        # Use a smaller n_fft for shorter signals
        n_fft = min(2048, len(audio) // 2)  # Ensure n_fft is at most half the signal length
        
        # Compute log-mel spectrogram
        mel_spec = librosa.feature.melspectrogram(y=audio, sr=sample_rate, n_fft=n_fft, n_mels=n_mels)
        log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)

        # Pad or truncate
        if log_mel_spec.shape[1] < max_pad_len:
            pad_width = max_pad_len - log_mel_spec.shape[1]
            log_mel_spec = np.pad(log_mel_spec, ((0, 0), (0, pad_width)), mode='constant')
        else:
            log_mel_spec = log_mel_spec[:, :max_pad_len]

        return log_mel_spec.T  # Shape: (174, 64)
    
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None


# Extract features
X_train_features = np.array([f for f in (extract_features(fp) for fp in train_file_paths) if f is not None])
X_test_features = np.array([f for f in (extract_features(fp) for fp in test_file_paths) if f is not None])

# Ensure labels match extracted features
y_train = to_categorical(y_train[:len(X_train_features)], num_classes=10)
y_test = to_categorical(y_test[:len(X_test_features)], num_classes=10)

# Normalize features
scaler = StandardScaler()
X_train_features = scaler.fit_transform(X_train_features.reshape(-1, X_train_features.shape[-1])).reshape(X_train_features.shape)
X_test_features = scaler.transform(X_test_features.reshape(-1, X_test_features.shape[-1])).reshape(X_test_features.shape)
class TransformerBlock(tf.keras.layers.Layer):
    """Transformer block with Multi-Head Attention and Feedforward Network."""
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation="relu"),
            tf.keras.layers.Dense(embed_dim)
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_model(input_shape, embed_dim=64, num_heads=4, ff_dim=128, num_classes=10):
    """CNN + Transformer hybrid model."""
    inputs = tf.keras.Input(shape=input_shape)

    # CNN Block
    x = tf.keras.layers.Conv1D(filters=64, kernel_size=3, activation="relu", padding="same")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Conv1D(filters=128, kernel_size=3, activation="relu", padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2)(x)

    # Linear embedding layer
    x = tf.keras.layers.Dense(embed_dim)(x)

    # Transformer Blocks
    x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)
    x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)

    # Global Average Pooling
    x = tf.keras.layers.GlobalAveragePooling1D()(x)

    # Fully Connected Layers
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    # Compile Model
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    
    return model

# Build the model
input_shape = (X_train_features.shape[1], X_train_features.shape[2])
model = build_model(input_shape)
# Learning rate scheduler
lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

# Early stopping
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
history = model.fit(X_train_features, y_train,
                    validation_data=(X_test_features, y_test),
                    epochs=50, batch_size=32, verbose=1,
                    callbacks=[early_stopping, lr_schedule])
# Evaluate the model
score = model.evaluate(X_test_features, y_test)
print(f"Transformer Model Accuracy: {score[1] * 100:.2f}%")


C:\Users\gargk\AppData\Roaming\Python\Python312\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1103
  warnings.warn(
C:\Users\gargk\AppData\Roaming\Python\Python312\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1323
  warnings.warn(


Epoch 1/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 155s 496ms/step - accuracy: 0.3467 - loss: 1.8041 - val_accuracy: 0.5220 - val_loss: 1.3493 - learning_rate: 0.0010
Epoch 2/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 68s 306ms/step - accuracy: 0.5315 - loss: 1.3446 - val_accuracy: 0.6543 - val_loss: 1.0652 - learning_rate: 0.0010
Epoch 3/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 65s 295ms/step - accuracy: 0.6082 - loss: 1.1368 - val_accuracy: 0.6400 - val_loss: 1.0651 - learning_rate: 0.0010
Epoch 4/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 63s 289ms/step - accuracy: 0.6531 - loss: 1.0141 - val_accuracy: 0.6829 - val_loss: 0.9294 - learning_rate: 0.0010
Epoch 5/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 77s 353ms/step - accuracy: 0.6828 - loss: 0.9442 - val_accuracy: 0.7212 - val_loss: 0.8404 - learning_rate: 0.0010
Epoch 6/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 54s 243ms/step - accuracy: 0.7081 - loss: 0.8704 - val_accuracy: 0.7041 - val_loss: 0.8596 - learning_rate: 0.0010
Epoch 7/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 48s 218ms/step - accuracy: 0.

In [6]:
import os
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load metadata
metadata_path = "./archive/UrbanSound8K.csv"
metadata = pd.read_csv(metadata_path)

# Create file paths and labels
file_paths = ["./archive/fold{}/{}".format(row['fold'], row["slice_file_name"]) for _, row in metadata.iterrows()]
labels = metadata["classID"].values

# Split into train and test sets
train_file_paths, test_file_paths, y_train, y_test = train_test_split(file_paths, labels, test_size=0.2, random_state=42)

# Feature extraction function
def extract_features(file_path, n_mfcc=40, max_pad_len=174):
    try:
        audio, sample_rate = librosa.load(file_path, sr=22050)
        
        # Handle n_fft warning by ensuring n_fft <= len(audio)
        n_fft = min(2048, len(audio))
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=n_mfcc, n_fft=n_fft)

        # Pad or truncate to fixed shape
        if mfccs.shape[1] < max_pad_len:
            pad_width = max_pad_len - mfccs.shape[1]
            mfccs = np.pad(mfccs, ((0, 0), (0, pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :max_pad_len]

        return mfccs.T  # Shape: (174, 40)
    
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Apply feature extraction
X_train_features = np.array([f for f in (extract_features(fp) for fp in train_file_paths) if f is not None])
X_test_features = np.array([f for f in (extract_features(fp) for fp in test_file_paths) if f is not None])

# Ensure no empty feature sets
if X_train_features.size == 0 or X_test_features.size == 0:
    raise ValueError("Feature extraction failed for all files. Check file paths and processing logic.")

# Normalize features
scaler = StandardScaler()
X_train_features = scaler.fit_transform(X_train_features.reshape(-1, X_train_features.shape[-1])).reshape(X_train_features.shape)
X_test_features = scaler.transform(X_test_features.reshape(-1, X_test_features.shape[-1])).reshape(X_test_features.shape)

# Ensure labels are one-hot encoded
y_train = to_categorical(y_train[:len(X_train_features)], num_classes=10)
y_test = to_categorical(y_test[:len(X_test_features)], num_classes=10)

# Transformer-based model
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation="relu"),
            tf.keras.layers.Dense(embed_dim)
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, inputs, training=False):  # Fixed: Ensure training argument is included
        attn_output = self.att(inputs, inputs, training=training)  # Pass training
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Model function
def build_transformer_model(input_shape, embed_dim=64, num_heads=4, ff_dim=128, num_classes=10):
    inputs = tf.keras.Input(shape=input_shape)
    
    # Linear embedding layer
    x = tf.keras.layers.Dense(embed_dim)(inputs)
    
    # Transformer blocks
    x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)
    x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)

    # Global average pooling
    x = tf.keras.layers.GlobalAveragePooling1D()(x)

    # Dense layers
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="categorical_crossentropy", metrics=["accuracy"])
    
    return model

# Build the transformer model
input_shape = (X_train_features.shape[1], X_train_features.shape[2])
transformer_model = build_transformer_model(input_shape)

# Train the model
history = transformer_model.fit(X_train_features, y_train, validation_data=(X_test_features, y_test),
                                epochs=50, batch_size=32, verbose=1)

# Evaluate the model
score = transformer_model.evaluate(X_test_features, y_test)
print(f"Transformer Model Accuracy: {score[1] * 100:.2f}%")


Epoch 1/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 138s 499ms/step - accuracy: 0.4481 - loss: 1.6062 - val_accuracy: 0.7281 - val_loss: 0.8193
Epoch 2/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 109s 497ms/step - accuracy: 0.7452 - loss: 0.7765 - val_accuracy: 0.7894 - val_loss: 0.6101
Epoch 3/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 93s 425ms/step - accuracy: 0.8275 - loss: 0.5418 - val_accuracy: 0.8420 - val_loss: 0.4713
Epoch 4/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 99s 450ms/step - accuracy: 0.8733 - loss: 0.3948 - val_accuracy: 0.8483 - val_loss: 0.4771
Epoch 5/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 98s 449ms/step - accuracy: 0.8847 - loss: 0.3467 - val_accuracy: 0.8764 - val_loss: 0.3778
Epoch 6/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 96s 436ms/step - accuracy: 0.9152 - loss: 0.2590 - val_accuracy: 0.8809 - val_loss: 0.3746
Epoch 7/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 94s 427ms/step - accuracy: 0.9188 - loss: 0.2452 - val_accuracy: 0.8930 - val_loss: 0.3348
Epoch 8/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 92s 419ms/step - accuracy: 0.9216 - loss: